In [1]:
import pandas as pd

# INPUTS
SEL_PATH = "selected_cpgs_final_5000.txt"
HORV_PATH = "Horvath_353_clock_CpGs.csv"

# 1) Read selected 5000 (one CpG per line)
with open(SEL_PATH, "r") as f:
    selected = {line.strip() for line in f if line.strip()}

# 2) Read Horvath 353 (CSV with column 'cpg')
horvath = set(pd.read_csv(HORV_PATH)["cpg"].astype(str).str.strip())

# 3) Intersection
common = selected & horvath

print(f"Selected CpGs: {len(selected)}")
print(f"Horvath 353 CpGs: {len(horvath)}")
print(f"Common CpGs: {len(common)}")


Selected CpGs: 4834
Horvath 353 CpGs: 353
Common CpGs: 3


In [2]:
# PATH AI FILE
f_69914  = "selected_cpgs_final_69914.txt"
f_225845 = "selected_cpgs_final_225845.txt"
f_287331 = "selected_cpgs_final_287331.txt"

def load_cpgs(path):
    with open(path) as f:
        return set(line.strip() for line in f if line.strip())

S_69914  = load_cpgs(f_69914)
S_225845 = load_cpgs(f_225845)
S_287331 = load_cpgs(f_287331)

print(f"GSE69914:  {len(S_69914)} CpGs")
print(f"GSE225845: {len(S_225845)} CpGs")
print(f"GSE287331: {len(S_287331)} CpGs")

# -------------------------
# INTERSEZIONE DI TUTTI E 3
# -------------------------
I_3 = S_69914 & S_225845 & S_287331
print("\nCpGs comuni a TUTTI e TRE:", len(I_3))

# -------------------------
# INTERSEZIONI A DUE A DUE
# -------------------------
I_69914_225845 = S_69914 & S_225845
I_69914_287331 = S_69914 & S_287331
I_225845_287331 = S_225845 & S_287331

print("\nCpGs comuni a due a due:")
print(f"  GSE69914 ∩ GSE225845: {len(I_69914_225845)}")
print(f"  GSE69914 ∩ GSE287331: {len(I_69914_287331)}")
print(f"  GSE225845 ∩ GSE287331: {len(I_225845_287331)}")


GSE69914:  5000 CpGs
GSE225845: 5000 CpGs
GSE287331: 5000 CpGs

CpGs comuni a TUTTI e TRE: 3

CpGs comuni a due a due:
  GSE69914 ∩ GSE225845: 84
  GSE69914 ∩ GSE287331: 24
  GSE225845 ∩ GSE287331: 185


In [1]:
# ============================================================
# CONFRONTO CROSS-DATASET — geni delle 50 CpG
# Runnare DOPO aver eseguito annotate_50cpg.py sui tre dataset
#
# Prima di runnare questa cella devi avere salvato m50 per ogni
# dataset con nome diverso. Aggiungi alla fine di annotate_50cpg.py:
#
#   m50_ds1 = m50.copy()   # dopo aver runnato sul dataset 1
#   m50_ds2 = m50.copy()   # dopo aver runnato sul dataset 2
#   m50_ds3 = m50.copy()   # dopo aver runnato sul dataset 3
#
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUTDIR = Path("./post_fs_annotation")
OUTDIR.mkdir(parents=True, exist_ok=True)

m50_ds1 = pd.read_csv("50cpg_annotated_GSE69914.tsv", sep="\t")
m50_ds2 = pd.read_csv("50cpg_annotated_GSE225845.tsv", sep="\t")
m50_ds3 = pd.read_csv("50cpg_annotated_GSE287331.tsv", sep="\t")

# ── 1. Estrai geni per direzione da ogni dataset ──
def get_genes_by_direction(m50_df, direction):
    return set(
        m50_df[m50_df["direction"] == direction]["GENE_SYMBOL"].astype(str)
    )

# Dataset 1 — prevalenza ipometilazione
ds1_hypo  = get_genes_by_direction(m50_ds1, "hypomethylated_in_adj")
ds1_hyper = get_genes_by_direction(m50_ds1, "hypermethylated_in_adj")
ds1_all   = ds1_hypo | ds1_hyper

# Dataset 2
ds2_hypo  = get_genes_by_direction(m50_ds2, "hypomethylated_in_adj")
ds2_hyper = get_genes_by_direction(m50_ds2, "hypermethylated_in_adj")
ds2_all   = ds2_hypo | ds2_hyper

# Dataset 3
ds3_hypo  = get_genes_by_direction(m50_ds3, "hypomethylated_in_adj")
ds3_hyper = get_genes_by_direction(m50_ds3, "hypermethylated_in_adj")
ds3_all   = ds3_hypo | ds3_hyper

print(f"Geni unici DS1: {len(ds1_all)}")
print(f"Geni unici DS2: {len(ds2_all)}")
print(f"Geni unici DS3: {len(ds3_all)}")

# ── 2. Intersezioni (qualsiasi direzione) ──
print(f"\n--- Intersezioni geni (qualsiasi direzione) ---")
print(f"DS1 ∩ DS2       : {sorted(ds1_all & ds2_all)}")
print(f"DS1 ∩ DS3       : {sorted(ds1_all & ds3_all)}")
print(f"DS2 ∩ DS3       : {sorted(ds2_all & ds3_all)}")
print(f"DS1 ∩ DS2 ∩ DS3 : {sorted(ds1_all & ds2_all & ds3_all)}")

# ── 3. Intersezioni per STESSA direzione (biologicamente più forte) ──
print(f"\n--- Intersezioni STESSA direzione ---")
# hypo in tutti e due/tre
h12 = ds1_hypo & ds2_hypo
h13 = ds1_hypo & ds3_hypo
h23 = ds2_hypo & ds3_hypo
h123 = ds1_hypo & ds2_hypo & ds3_hypo
print(f"Hypo DS1∩DS2    : {sorted(h12)}")
print(f"Hypo DS1∩DS3    : {sorted(h13)}")
print(f"Hypo DS2∩DS3    : {sorted(h23)}")
print(f"Hypo DS1∩DS2∩DS3: {sorted(h123)}")

H12 = ds1_hyper & ds2_hyper
H13 = ds1_hyper & ds3_hyper
H23 = ds2_hyper & ds3_hyper
H123 = ds1_hyper & ds2_hyper & ds3_hyper
print(f"Hyper DS1∩DS2   : {sorted(H12)}")
print(f"Hyper DS1∩DS3   : {sorted(H13)}")
print(f"Hyper DS2∩DS3   : {sorted(H23)}")
print(f"Hyper DS1∩DS2∩DS3:{sorted(H123)}")

# ── 4. Geni con direzione DISCORDANTE tra dataset ──
print(f"\n--- Geni con direzione DISCORDANTE (interessante biologicamente) ---")
# gene hypo in DS1 ma hyper in DS2 o DS3
discordant_12 = ds1_hypo & ds2_hyper  # hypo in 1, hyper in 2
discordant_21 = ds1_hyper & ds2_hypo
discordant_13 = ds1_hypo & ds3_hyper
discordant_31 = ds1_hyper & ds3_hypo
discordant_23 = ds2_hypo & ds3_hyper
discordant_32 = ds2_hyper & ds3_hypo

for label, s in [
    ("hypo DS1 / hyper DS2", discordant_12),
    ("hyper DS1 / hypo DS2", discordant_21),
    ("hypo DS1 / hyper DS3", discordant_13),
    ("hyper DS1 / hypo DS3", discordant_31),
    ("hypo DS2 / hyper DS3", discordant_23),
    ("hyper DS2 / hypo DS3", discordant_32),
]:
    if s:
        print(f"  {label}: {sorted(s)}")

# ── 5. Tabella riassuntiva per tesi ──
all_genes = ds1_all | ds2_all | ds3_all
rows = []
for g in sorted(all_genes):
    d1 = ("hypo" if g in ds1_hypo else "hyper" if g in ds1_hyper else "-")
    d2 = ("hypo" if g in ds2_hypo else "hyper" if g in ds2_hyper else "-")
    d3 = ("hypo" if g in ds3_hypo else "hyper" if g in ds3_hyper else "-")
    n_present = sum([g in ds1_all, g in ds2_all, g in ds3_all])
    concordant = (
        d1 == d2 == d3 if n_present == 3
        else d1 == d2 if (n_present == 2 and g in ds1_all and g in ds2_all)
        else d1 == d3 if (n_present == 2 and g in ds1_all and g in ds3_all)
        else d2 == d3 if (n_present == 2 and g in ds2_all and g in ds3_all)
        else True
    )
    rows.append({
        "gene": g,
        "DS1": d1, "DS2": d2, "DS3": d3,
        "n_datasets": n_present,
        "concordant": concordant,
    })

df_summary = pd.DataFrame(rows)

# mostra solo geni presenti in almeno 2 dataset
df_multi = df_summary[df_summary["n_datasets"] >= 2].sort_values(
    ["n_datasets", "concordant"], ascending=[False, False]
)
print(f"\n--- Geni presenti in ≥2 dataset ---")
if len(df_multi) == 0:
    print("Nessun gene in comune tra dataset.")
    print("Questo conferma eterogeneità di coorte — atteso nel cross-dataset.")
else:
    print(df_multi.to_string(index=False))

df_summary.to_csv(OUTDIR / "cross_dataset_gene_summary.tsv", sep="\t", index=False)
print(f"\nSalvato: {OUTDIR / 'cross_dataset_gene_summary.tsv'}")

Geni unici DS1: 37
Geni unici DS2: 45
Geni unici DS3: 46

--- Intersezioni geni (qualsiasi direzione) ---
DS1 ∩ DS2       : []
DS1 ∩ DS3       : []
DS2 ∩ DS3       : ['IRX4']
DS1 ∩ DS2 ∩ DS3 : []

--- Intersezioni STESSA direzione ---
Hypo DS1∩DS2    : []
Hypo DS1∩DS3    : []
Hypo DS2∩DS3    : []
Hypo DS1∩DS2∩DS3: []
Hyper DS1∩DS2   : []
Hyper DS1∩DS3   : []
Hyper DS2∩DS3   : []
Hyper DS1∩DS2∩DS3:[]

--- Geni con direzione DISCORDANTE (interessante biologicamente) ---
  hyper DS2 / hypo DS3: ['IRX4']

--- Geni presenti in ≥2 dataset ---
gene DS1   DS2  DS3  n_datasets  concordant
IRX4   - hyper hypo           2       False

Salvato: post_fs_annotation\cross_dataset_gene_summary.tsv


In [2]:
import polars as pl
from pathlib import Path

# --- FILE PATHS ---
file_353 = Path("final_cpgs_K5000_gse287331_225845_fs_v1_20260303_152925.txt")
file_50  = Path("final_50_cpgs.csv")

# --- LOAD 353 CpGs (TXT) ---
cpgs_353 = {
    line.strip()
    for line in file_353.read_text().splitlines()
    if line.strip()
}

# --- LOAD 50 CpGs (CSV) ---
cpgs_50 = set(
    pl.read_csv(file_50)["CpG_ID"]
    .drop_nulls()
    .to_list()
)

# --- INTERSECTION ---
common_cpgs = sorted(cpgs_353 & cpgs_50)

print("Horvath 353 CpGs:", len(cpgs_353))
print("Final 50 CpGs:", len(cpgs_50))
print("CpGs in common:", len(common_cpgs))
print("Common CpGs:", common_cpgs)

Horvath 353 CpGs: 5000
Final 50 CpGs: 50
CpGs in common: 50
Common CpGs: ['cg00066511', 'cg00733150', 'cg00974835', 'cg01535733', 'cg01720705', 'cg02746232', 'cg03896694', 'cg03994018', 'cg04279629', 'cg07743843', 'cg08198486', 'cg08206308', 'cg09108491', 'cg09413529', 'cg09649046', 'cg10360552', 'cg12504415', 'cg13184823', 'cg13246583', 'cg13431205', 'cg13608166', 'cg14473974', 'cg14692804', 'cg14988425', 'cg15620385', 'cg15700739', 'cg15922085', 'cg15957959', 'cg16652651', 'cg16879549', 'cg17334114', 'cg17561417', 'cg18023339', 'cg18481241', 'cg18546668', 'cg18928066', 'cg19733867', 'cg20025656', 'cg21233690', 'cg21511523', 'cg22635491', 'cg22697574', 'cg22864266', 'cg23303074', 'cg24443925', 'cg24751928', 'cg24779381', 'cg25508633', 'cg26048006', 'cg27034924']
